#### **Association Rules and Apriori Algorithm**

**SIMPLE END-TO-END MARKET BASKET RECOMMENDER**

`From a transaction file -> frequent itemset -> rules -> recommendations`
Needs:  `pandas, mlxtend (pip install pandas mlxtend)`

Needs the file: grocery_transactions.csv (in the same folder as this notebook)

In [52]:
import pandas as pd                                     # tables (Dataframes)
from mlxtend.preprocessing import TransactionEncoder    # baskets -> True/False table
from mlxtend.frequent_patterns import apriori           # finds the frequent itemsets
from mlxtend.frequent_patterns import association_rules        # returns itemset into rulles

In [53]:
# STEP 1 - READ THE TRANSACTION FILE
'''
The file has ONE ROW PER LINE ITEM (not one row per basket). 
Several rows share the same transaction_id: those rows together are one receipt.

'''
file_path = 'grocery_transactions.csv'
df = pd.read_csv(file_path)

print('STEP 1 - file loaded')
print(f'    rows (line items)   : {len(df):,}')
print(f'    transactions        : {df["transaction_id"].nunique():,}') # nunique() = how many distinct values
print(f'    columns             : {list(df.columns)}\n')
 

STEP 1 - file loaded
    rows (line items)   : 13,234
    transactions        : 2,472
    columns             : ['transaction_id', 'timestamp', 'customer_id', 'item', 'category', 'quantity', 'unit_price', 'line_total']



In [54]:
# STEP 2 - ORGANIZE THE DATA: one list of items per transaction

'''
2a) Clean the product names first
- .str.strip() removes spaces at the start and the end ("    butter " -> "butter")
- .str.lower() puts everything in lowercase            ("BUTTER" -> "butter")
without this, "butter", "BUTTER" and " butter " would count as 3 products.
'''
df['item'] = df['item'].str.strip().str.lower()

'''
2b) Group the rows by receipt and collect the item of each receipt.
- gather the item values of each receipt
- run a small function on each group
- remove duplicates inside the same basket (an item scanned twice is still INE item)
- put them in a fixed order (easier to read)

'''

baskets = (df.groupby('transaction_id')['item'].apply(lambda items: sorted(set(items))).tolist() 
           ) # lambda = a one-line function, # .tolist() -> a plain pyhton list 

print('STEP 2 - data organized into baskets')
print(f'    number of baskets: {len(baskets):,}')
print(f'    first 3 baskets     :')
for b in baskets[:7]:
    print('     ', b)
print()


STEP 2 - data organized into baskets
    number of baskets: 2,472
    first 3 baskets     :
      ['eggs', 'parmesan']
      ['baby formula', 'beer', 'chicken breast', 'diapers', 'red wine']
      ['avocado', 'berries', 'granola', 'peanuts', 'whole wheat bread', 'yogurt']
      ['beer', 'berries', 'lettuce', 'tomatoes', 'yogurt']
      ['ground beef', 'orange juice', 'pasta', 'tomato sauce', 'trash bags', 'whole wheat bread']
      ['conditioner', 'salmon', 'shampoo', 'toothpaste', 'whole wheat bread']
      ['eggs', 'ice cream', 'lettuce', 'white bread']



In [55]:
# STEP 3 -  ENCODE THE BASKETS AS A TRUE/FALSE TABLE

'''
Apriori cannot read lists of words; it needs a table with one row per basket, one colum per product, True where the producr is present.

 TransactionEncoder()   : the translator object
 - .fit(baskets)        : LEARN the list of all products (these become the columns)
 - .transform(baskets)  : APPLY it -> the True/False array
 - te.columns_          : the product names it learned, in column order
'''
te = TransactionEncoder()
translated_data = te.fit(baskets).transform(baskets) # We obtain a True/False-like Matrix
basket_table = pd.DataFrame(translated_data, columns=te.columns_)
# basket_table.to_csv('true_false_data.csv')
print('STEP  3 - basket table built')
print(f'    shape: {basket_table.shape}     (baskets x products)\n')
print(basket_table.iloc[:3, :5]) # .iloc[rows, cols] -> first 3 rows, first 5 columns
print()


STEP  3 - basket table built
    shape: (2472, 50)     (baskets x products)

   apples  avocado  baby formula  baby wipes  bacon
0   False    False         False       False  False
1   False    False          True       False  False
2   False     True         False       False  False



In [63]:
# STEP 4 - APRIORI: FIND THE FREQUENT ITEMSETS
'''
apriori() keeps only the combinations of products that appear often enough.

- basket_table          the True/False table from STEP 3
- min_support=0.02      KEEP an itemset only if it appears in at least 2% of baskets.
                        Support is a FRACTION: 0.02 x 2472 baskets = about 50 baskets.
                        Higher value -> fewer, stronger itemsets and a faster run
                        Lower value -> more itemsets, including coincidences.
- use_colnames          show PRODUCT NAMES in the result instead of column numbers.
                        ALWAYS use True: without it you get frozenset({7, 19})
- max_len=3             stop at combinations of 3 products. Longer ones are rare and
                        hard to act on. Set it to None for no limit
                        
'''
frequent_itemsets = apriori(basket_table,
                            min_support=0.02,
                            use_colnames=True,
                            max_len=3
                            )

# The rsult has 2 columns: 'support' {the fraction} and 'itemsets' (the products)
# Add one helper column: how many product are in each itemset

frequent_itemsets['size'] = frequent_itemsets['itemsets'].apply(len) # apply(len) = length of each set

print('STEP 4 - frequent itemsets found by Apriori')
print(f'    total itemsets : {len(frequent_itemsets)}\n')
by_size = frequent_itemsets["size"].value_counts().sort_index()
print(f'    by size        : {by_size.to_dict()}')
print(' \nthe 5 most frequent itemsets of 2 products:')
pairs = frequent_itemsets[frequent_itemsets['size']==2].nlargest(5, 'support') #nlargest = top rows
print()
for _, row in pairs.iterrows():
    print(f'    {row["support"]:.3f}    {set(row["itemsets"])}')
print()
    


# trash = frequent_itemsets
# trash['itemsets'] = trash['itemsets'].apply(lambda x: ', '.join(list(x)))
# trash


STEP 4 - frequent itemsets found by Apriori
    total itemsets : 184

    by size        : {1: 50, 2: 86, 3: 48}
 
the 5 most frequent itemsets of 2 products:

    0.104    {'pasta', 'tomato sauce'}
    0.102    {'white bread', 'butter'}
    0.096    {'coffee beans', 'milk 2%'}
    0.082    {'white bread', 'strawberry jam'}
    0.082    {'parmesan', 'pasta'}



In [57]:
my_list = ['Steve', 'Hamza', 'Marius', 'Hamza']
print(my_list)
my_set = set(my_list)
print(my_set)
my_sorted = sorted(my_set)
print(my_sorted)

['Steve', 'Hamza', 'Marius', 'Hamza']
{'Hamza', 'Steve', 'Marius'}
['Hamza', 'Marius', 'Steve']


T1 -> bread, tea
T1 -> bread
T1 -> tea
T2 -> sugar, bulb
T2 -> sugar
T2 -> bulb
T3 -> tea, sugar
...

Item = {bread, tea, sugar, bulb}

TransactionEncoder

      bread   tea    sugar    bulb
T1.    True.   True   False     False

T2.    False  False    True     True

T3.    False    True    True    False




